In [12]:
import os
import json

fp = os.path.join('data','matches','mw2', 'imbabura_delfin_statistics.json')

with open(fp, 'r') as f:
    data = json.load(f)

groups = data['statistics'][0]['groups']

In [13]:
for group in groups:
        print(group['groupName'])

Possession
Shots
TVData
Shots extra
Passes
Duels
Defending


In [14]:
groups[0]['statisticsItems']

[{'name': 'Ball possession',
  'home': '67%',
  'away': '33%',
  'compareCode': 1,
  'statisticsType': 'positive',
  'valueType': 'event',
  'homeValue': 67,
  'awayValue': 33}]

In [15]:
import json
import os
import pandas as pd

# Find all matchweek folders
base_dir = './data/matches'
matchweek_dirs = os.listdir(base_dir)
# Select only directories
matchweek_folders = [x for x in matchweek_dirs if os.path.isdir(os.path.join(base_dir,x))]

match_files = []
final_data = []
# Loop through folders
for week in matchweek_folders:
  match_files = os.listdir(os.path.join(base_dir, week))
  # Loop through files
  for file in match_files:
         # Get team names and filetype to only use 'statistics' files
         home, away, filetype = file.split('_')

         if filetype == 'statistics.json':
            match_fp = os.path.join(base_dir, week, file)
            # Read 'statistics' file
            with open(match_fp, 'r') as f:
               data = json.load(f)

            groups = data['statistics'][0]['groups']


            # Create one row for each team for each match
            for team in ['home', 'away']:
                data = {'home': home,
                        'away': away,
                        'team': home if team=='home' else away,
                        'possesion': groups[0]['statisticsItems'][0][team],

                        # Shots
                        'total_shots': groups[1]['statisticsItems'][0][team],
                        'on_target': groups[1]['statisticsItems'][1][team],
                        'off_target': groups[1]['statisticsItems'][2][team],
                        'blocket_shots': groups[1]['statisticsItems'][3][team],
                        'shots_inside_box': groups[3]['statisticsItems'][4][team],
                        'shots_outside_box': groups[3]['statisticsItems'][5][team],
                        # Defensive
                        'gk_saves': groups[3]['statisticsItems'][6][team],
                        }
                final_data.append(data)

    


pd.DataFrame(final_data)

IndexError: list index out of range

## Lets do some data quality checks

### Lineups

In [ ]:
import os
import pandas as pd
import re
import json


def extract_json_from_html(html_path, save_output=False):
    html_file = open(html_path, 'r')
    html = html_file.read()
    html_file.close()
    regex_pattern = r'(?<=require\.config\.params\["args"\].=.)[\s\S]*?;'
    data_txt = re.findall(regex_pattern, html)[0]

    # add quotations for json parser
    data_txt = data_txt.replace('matchId', '"matchId"')
    data_txt = data_txt.replace('matchCentreData', '"matchCentreData"')
    data_txt = data_txt.replace('matchCentreEventTypeJson', '"matchCentreEventTypeJson"')
    data_txt = data_txt.replace('formationIdNameMappings', '"formationIdNameMappings"')
    data_txt = data_txt.replace('};', '}')

    if save_output:
        # save json data to txt
        output_file = open(f"{html_path}.txt", "wt")
        n = output_file.write(data_txt)
        output_file.close()

    return data_txt

def extract_data_from_dict(data):
    # load data from json
    event_types_json = data["matchCentreEventTypeJson"]
    formation_mappings = data["formationIdNameMappings"]
    events_dict = data["matchCentreData"]["events"]
    teams_dict = {data["matchCentreData"]['home']['teamId']: data["matchCentreData"]['home']['name'],
                  data["matchCentreData"]['away']['teamId']: data["matchCentreData"]['away']['name']}
    players_dict = data["matchCentreData"]["playerIdNameDictionary"]
    # create players dataframe
    players_home_df = pd.DataFrame(data["matchCentreData"]['home']['players'])
    players_home_df["teamId"] = data["matchCentreData"]['home']['teamId']
    players_away_df = pd.DataFrame(data["matchCentreData"]['away']['players'])
    players_away_df["teamId"] = data["matchCentreData"]['away']['teamId']
    players_df = pd.concat([players_home_df, players_away_df])
    players_ids = data["matchCentreData"]["playerIdNameDictionary"]
    return events_dict, players_df, teams_dict

In [ ]:
import pandas as pd
home = 'catolica'
away = 'cumbaya'
mw = 10

lineups = 'data/matches/mw7/delfin_catolica_lineups.json'
df = parse_json_lineups(file = f'{home}_{away}_lineups.json', base_dir = 'data/matches', week = f'mw{mw}')
df.head()

,totalPass,accuratePass,totalLongBalls,accurateLongBalls,totalClearance,savedShotsFromInsideTheBox,saves,punches,totalKeeperSweeper,accurateKeeperSweeper,...,totalOffside,bigChanceMissed,ShotOnTarget,goals,goodHighClaim,clearanceOffLine,TotalShots,accuratePass_p90,TotalShots_p90,ShotOnTarget_p90
0,38.0,32.0,11.0,5.0,3.0,1.0,2.0,1.0,2.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,32.0,0.0,0.0
1,36.0,25.0,5.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,25.0,0.0,0.0
2,44.0,38.0,3.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38.0,0.0,0.0
3,67.0,54.0,7.0,3.0,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,54.0,1.0,0.0
4,49.0,44.0,2.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,44.0,0.0,0.0


In [ ]:
df.columns

Index(['totalPass', 'accuratePass', 'totalLongBalls', 'accurateLongBalls',
       'totalClearance', 'savedShotsFromInsideTheBox', 'saves', 'punches',
       'totalKeeperSweeper', 'accurateKeeperSweeper', 'minutesPlayed',
       'touches', 'rating', 'possessionLostCtrl', 'ratingVersions', 'player',
       'team', 'home', 'away', 'matchweek', 'totalCross', 'accurateCross',
       'aerialWon', 'duelLost', 'duelWon', 'totalContest', 'wonContest',
       'bigChanceCreated', 'interceptionWon', 'wasFouled', 'fouls', 'keyPass',
       'aerialLost', 'outfielderBlock', 'totalTackle', 'ShotOffTarget',
       'blockedScoringAttempt', 'challengeLost', 'dispossessed', 'goalAssist',
       'totalOffside', 'bigChanceMissed', 'ShotOnTarget', 'goals',
       'goodHighClaim', 'clearanceOffLine', 'TotalShots', 'accuratePass_p90',
       'TotalShots_p90', 'ShotOnTarget_p90'],
      dtype='object')

In [ ]:
df_lineups = df[['team', 'TotalShots', 'ShotOnTarget', 'goals', 'ShotOffTarget', 'blockedScoringAttempt']].groupby('team').sum()
df_lineups

,TotalShots,ShotOnTarget,goals,ShotOffTarget,blockedScoringAttempt
team,,,,,
catolica,12.0,6.0,2.0,6.0,6.0
cumbaya,9.0,3.0,1.0,6.0,5.0


This is the shooting/goals according to 'lineups'. It has to match the data in 'shotmap' and 'statistics'.

### Shotmap

In [ ]:
# !pip install inflection

In [ ]:
import os
import json
import pandas as pd
from inflection import underscore

def parse_json_shotmap(file, base_dir, week):
    
    home, away, filetype = file.split('_')
    final_data = []

    if filetype == 'shotmap.json':
        # Read 'statistics' file
        with open(os.path.join(base_dir, week, file), 'r') as f:
            shots = json.load(f)['shotmap']
    
        # Create one row for each shot
    
        for shot in shots:
            data = {'home': home,
                    'away': away,
                    'team': home if shot['isHome'] else away,
                    'matchweek': week[-1]
                    }
            for info in shot:
                if info == 'player':
                    data['player'] = shot['player']['name']
                # If stat is a dict, create one stat per each dict key
                elif type(shot[info]) == dict:
                    for key in shot[info]:
                        # If stat is a dict, create one stat per each dict key
                        if type(shot[info][key]) == dict:
                            for key2 in shot[info][key]:
                                data[f"{underscore(info)}_{key}_{key2}"] = shot[info][key][key2]
                                # print(shot[info][key][key2])
    
                        else:
                            data[f"{underscore(info)}_{key}"] = shot[info][key]
                        # print(shot[info][key])
                elif info != 'isHome':
                    data[underscore(info)] = shot[info]
                    # print(shot[info])
    
            final_data.append(data)

    df = pd.DataFrame(final_data).fillna(0)
    return df

In [ ]:
fp = 'data/matches/mw7/delfin_catolica_shotmap.json'

df = parse_json_shotmap(file = f'{home}_{away}_shotmap.json', base_dir = 'data/matches', week = f'mw{mw}')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/matches\\mw10\\catolica_cumbaya_shotmap.json'

In [ ]:
df.columns

Index(['home', 'away', 'team', 'matchweek', 'player', 'shot_type', 'situation',
       'player_coordinates_x', 'player_coordinates_y', 'player_coordinates_z',
       'body_part', 'goal_mouth_location', 'goal_mouth_coordinates_x',
       'goal_mouth_coordinates_y', 'goal_mouth_coordinates_z',
       'block_coordinates_x', 'block_coordinates_y', 'block_coordinates_z',
       'id', 'time', 'added_time', 'time_seconds', 'draw_start_x',
       'draw_start_y', 'draw_block_x', 'draw_block_y', 'draw_end_x',
       'draw_end_y', 'draw_goal_x', 'draw_goal_y', 'reversed_period_time',
       'reversed_period_time_seconds', 'incident_type', 'goal_type'],
      dtype='object')

In [ ]:
df['shot_type'].unique()

array(['block', 'save', 'miss', 'goal'], dtype=object)

Data from Shotmap

In [ ]:
# Goals
df1 = df[['team', 'shot_type']][df['shot_type'] == 'goal'].groupby('team').count()
df1 = df1.rename(columns={"shot_type": "goals"})

# On Target
df2 = df[['team', 'shot_type']][df['shot_type'].isin(['save', 'goal'])].groupby('team').count()
df2 = df2.rename(columns={"shot_type": "ShotsOnTarget"})

# Shots
df3 = df[['team', 'shot_type']].groupby('team').count()
df3 = df3.rename(columns={"shot_type": "Shots"})

# Join df2 and df3
df_shotmap = df3.join(df2, how='outer')

# Join df and df1
df_shotmap = df_shotmap.join(df1, how='outer')

df_shotmap = df_shotmap.fillna(0)

df_shotmap

,Shots,ShotsOnTarget,goals
team,,,
cumbaya,8,4,0.0
emelec,13,5,2.0


### Statistics

In [ ]:
import os
import json
import pandas as pd

def parse_json_statistics(file, base_dir, week):
    home, away, filetype = file.split('_')

    final_data = []

    if filetype == 'statistics.json':
        # Read 'statistics' file
        with open(os.path.join(base_dir, week, file), 'r') as f:
            data = json.load(f)

        groups = data['statistics'][0]['groups']

        # Create one row for each team for each match
        for team in ['home', 'away']:
            data = {'home': home,
                    'away': away,
                    'team': home if team == 'home' else away,
                    'matchweek': week[2:]
                    }

            for stat_group in groups:
                for stat in stat_group['statisticsItems']:
                    stat_name = stat['name'].lower().replace(' ', '_')
                    data[stat_name] = stat[team+'Value']
            final_data.append(data)

    df = pd.DataFrame(final_data).fillna(0)

    # -------------------------- Passes
    # # Split passes column
    # df['accurate_passes'] = df['accurate_passes'].split().str[0]
    df.rename(columns={'accurate_passes': 'passes_completed'}, inplace=True)
    
    # # Change data type to numeric to be able to perform calculations
    # df['passes'] = pd.to_numeric(df['passes'])
    # df['passes_completed'] = pd.to_numeric(df['passes_completed'])
    
    # # Calculate new metric
    df.insert(df.columns.get_loc('passes') + 2, 'passes_accuracy',
              round(df['passes_completed'] / df['passes'], 4))
    
    
    # def separate_percentages(stat, df):
    #     accurate = f'{stat}_completed'
    #     accuracy = f'{stat}_accuracy'
    
    #     # Split column into two
    #     df[[stat, accurate]] = df[stat].str.split().str[0].str.split('/',
    #                                                                  expand=True)
    
    #     # Move new column next to old relevant column
    #     column_to_move = df.pop(accurate)
    #     df.insert(df.columns.get_loc(stat) + 1, accurate, column_to_move)
    
    #     # Change data type to numeric to be able to perform calculations
    #     df[stat] = pd.to_numeric(df[stat])
    #     df[accurate] = pd.to_numeric(df[accurate])
    
    #     # Calculate new metric
    #     df.insert(df.columns.get_loc(stat) + 2, accuracy,
    #               round(df[stat] / df[accurate], 4))
    #     return df
    
    
    # df = separate_percentages('long_balls', df)
    # df = separate_percentages('dribbles', df)
    # df = separate_percentages('crosses', df)
    
    # df['ball_possession'] = df['ball_possession'].str.replace('%', '')

    return df

In [ ]:
fp = 'data/matches/mw7/delfin_catolica_statistics.json'
df = parse_json_statistics(file = f'{home}_{away}_statistics.json', base_dir = 'data/matches', week = f'mw{mw}')
df.head()

,home,away,team,matchweek,ball_possession,total_shots,shots_on_target,shots_off_target,blocked_shots,corner_kicks,...,passes_accuracy,long_balls,crosses,dribbles,possession_lost,duels_won,aerials_won,tackles,interceptions,clearances
0,emelec,cumbaya,emelec,8,56,13,5,6,2,5,...,0.8498,24,1,2,125,33,12,8,12,9
1,emelec,cumbaya,cumbaya,8,44,8,4,3,1,3,...,0.8090,13,1,9,118,43,5,16,8,17


In [ ]:
df.columns

Index(['home', 'away', 'team', 'matchweek', 'ball_possession', 'total_shots',
       'shots_on_target', 'shots_off_target', 'blocked_shots', 'corner_kicks',
       'offsides', 'fouls', 'yellow_cards', 'free_kicks', 'throw-ins',
       'goal_kicks', 'big_chances', 'big_chances_missed', 'shots_inside_box',
       'shots_outside_box', 'goalkeeper_saves', 'passes', 'passes_completed',
       'passes_accuracy', 'long_balls', 'crosses', 'dribbles',
       'possession_lost', 'duels_won', 'aerials_won', 'tackles',
       'interceptions', 'clearances'],
      dtype='object')

In [ ]:
df_statistics = df[['team', 'total_shots', 'shots_on_target', 'ball_possession', 'passes', 'passes_completed', 'passes_accuracy']].groupby('team').sum()
df_statistics

,total_shots,shots_on_target,ball_possession,passes,passes_completed,passes_accuracy
team,,,,,,
cumbaya,8,4,44,377,305,0.8090
emelec,13,5,56,486,413,0.8498


## Data Comparison

### Shot on Target

In [ ]:
sot1 = df_lineups['ShotOnTarget'].astype(int)
print(sot1)

sot2 = df_shotmap['ShotsOnTarget'].astype(int)
print(sot2)

sot3 = df_statistics['shots_on_target'].astype(int)
print(sot3)

team
cumbaya    4
emelec     5
Name: ShotOnTarget, dtype: int32
team
cumbaya    4
emelec     5
Name: ShotsOnTarget, dtype: int32
team
cumbaya    4
emelec     5
Name: shots_on_target, dtype: int32


In [ ]:
print(sot1.equals(sot2))
print(sot1.equals(sot3))
print(sot2.equals(sot3))

True
True
True


### Total Shots

In [ ]:
# lineups adds data from each player so sum could be misleading, ignore for total shots
# shots1 = df_lineups['TotalShots'].astype(int)
# print(shots1)

shots2 = df_shotmap['Shots'].astype(int)
print(shots2)

shots3 = df_statistics['total_shots'].astype(int)
print(shots3)

team
cumbaya     8
emelec     13
Name: Shots, dtype: int32
team
cumbaya     8
emelec     13
Name: total_shots, dtype: int32


In [ ]:
print(shots2.equals(shots3))

True


### Goals

In [ ]:
goals1 = df_lineups['goals'].astype(int)
print(goals1)

goals2 = df_shotmap['goals'].astype(int)
print(goals2)


team
cumbaya    0
emelec     2
Name: goals, dtype: int32
team
cumbaya    0
emelec     2
Name: goals, dtype: int32


In [ ]:
print(goals1.equals(goals2))

True


## Possession

In [ ]:
df_statistics['ball_possession'].values.sum() == 100

True

# Will explore player ratings in a single match

Will explore top 3 rated players per team and will show stats according to position

In [1]:
# Lets borrow some code from match_report.py :

import sqlite3
import os
import pandas as pd

# Parse statistics for match
def read_db(table_name, home, away):
    """
    Load data
    """

    # Connect to the SQLite database
    db_path = os.path.join(os.getcwd(), 'data', 'liga_pro_ecuador.db')
    conn = sqlite3.connect(db_path)

    # Define the SQL query
    query = f"""
    SELECT * FROM {table_name}
    WHERE home == "{home}"
    AND away == "{away}";
    """

    # Read the data into a Pandas DataFrame
    df = pd.read_sql_query(query, conn)

    # Close the database connection
    conn.close()

    return df

In [91]:
home = 'delfin'
away = 'independiente'

df = read_db('lineups', home, away)
df = df.fillna(0.0)

df.head()

,matchweek,home,away,team,player,position,minutesPlayed,rating,ratingVersions,totalPass,...,timestamp,penaltyConceded,hitWoodwork,penaltyWon,punches,totalKeeperSweeper,accurateKeeperSweeper,penaltyMiss,penaltySave,clearanceOffLine
0,1,delfin,independiente,delfin,Edisson Recalde,G,90.0,6.5,"{'original': 6.5, 'alternative': 6.7}",20.0,...,2024-04-29 00:42:33.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,delfin,independiente,delfin,Josué Cuero,D,90.0,6.7,"{'original': 6.7, 'alternative': 6.7}",15.0,...,2024-04-29 00:42:33.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,delfin,independiente,delfin,Jefferson Nazareno,D,90.0,6.4,"{'original': 6.4, 'alternative': 6.4}",20.0,...,2024-04-29 00:42:33.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,delfin,independiente,delfin,Nicolás Goitea,D,90.0,6.9,"{'original': 6.9, 'alternative': 6.8}",25.0,...,2024-04-29 00:42:33.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,delfin,independiente,delfin,Juan Manuel Elordi,D,90.0,6.6,"{'original': 6.6, 'alternative': 6.5}",23.0,...,2024-04-29 00:42:33.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [92]:
df.columns

Index(['matchweek', 'home', 'away', 'team', 'player', 'position',
       'minutesPlayed', 'rating', 'ratingVersions', 'totalPass',
       'accuratePass', 'totalLongBalls', 'accurateLongBalls', 'totalClearance',
       'savedShotsFromInsideTheBox', 'saves', 'touches', 'possessionLostCtrl',
       'totalCross', 'aerialLost', 'aerialWon', 'duelLost', 'duelWon',
       'dispossessed', 'interceptionWon', 'totalTackle', 'wasFouled', 'fouls',
       'keyPass', 'challengeLost', 'ShotOffTarget', 'outfielderBlock',
       'totalContest', 'wonContest', 'accurateCross', 'bigChanceCreated',
       'ShotOnTarget', 'blockedScoringAttempt', 'bigChanceMissed',
       'totalOffside', 'goodHighClaim', 'goalAssist', 'goals', 'TotalShots',
       'accuratePass_p90', 'TotalShots_p90', 'ShotOnTarget_p90', 'timestamp',
       'penaltyConceded', 'hitWoodwork', 'penaltyWon', 'punches',
       'totalKeeperSweeper', 'accurateKeeperSweeper', 'penaltyMiss',
       'penaltySave', 'clearanceOffLine'],
      dtype='ob

Top 3 players by rating for each team

In [93]:
def get_stats_for_position(df):
    position = df['position']
    if position == 'G':
        data = df[['saves', 'savedShotsFromInsideTheBox', 'totalLongBalls', 'accurateLongBalls', 'goodHighClaim']]
        print(data)
    elif position == 'D':
        data = df[['duelWon', 'aerialWon']]
        print(data)
    elif position == 'M':
        data = df[['keyPass', 'duelWon', 'aerialWon', 'totalLongBalls', 'accurateLongBalls', 'goodHighClaim']]
        print(data)
    elif position == 'F':
        data = df[['ShotOnTarget', 'goals']]
        print(data)
        

home_df = df[df['team'] == df['home']]
away_df = df[df['team'] == df['away']]

home_df = home_df.sort_values(by='rating', ascending=False).iloc[:3]
home_df = home_df.reset_index(drop=True)

away_df = away_df.sort_values(by='rating',  ascending=False).iloc[:3]
away_df = away_df.reset_index(drop=True)


for i, row in home_df.iterrows():
    print(f"\n{i+1}- {row['player']} - {row['rating']}")
    get_stats_for_position(row)


1- Luis Castro - 7.1
keyPass              2.0
duelWon              7.0
aerialWon            3.0
totalLongBalls       6.0
accurateLongBalls    2.0
goodHighClaim        0.0
Name: 0, dtype: object

2- Nicolás Goitea - 6.9
duelWon      2.0
aerialWon    1.0
Name: 1, dtype: object

3- Maikel Reyes - 6.7
keyPass              0.0
duelWon              7.0
aerialWon            1.0
totalLongBalls       1.0
accurateLongBalls    0.0
goodHighClaim        0.0
Name: 2, dtype: object


Lets show the best ranked stats for each top rated player.

First, lets calculate ranking for a list of stats

In [100]:
player = 'Luis Castro'

ranks = []

home_df = df[df['team'] == df['home']].copy()
away_df = df[df['team'] == df['away']].copy()

print(player + ' ranks')

for stat in ['totalPass',
       'accuratePass', 'totalLongBalls', 'accurateLongBalls', 'totalClearance',
       'savedShotsFromInsideTheBox', 'saves', 'touches',
       'totalCross', 'accurateCross', 'aerialWon', 'duelWon',
       'dispossessed', 'interceptionWon', 'totalTackle', 'wasFouled',
       'keyPass', 'challengeLost', 'ShotOffTarget', 'outfielderBlock',
       'totalContest', 'wonContest', 'bigChanceCreated',
       'ShotOnTarget','bigChanceMissed',
       'totalOffside', 'goodHighClaim', 'goalAssist', 'goals', 'TotalShots',
       'accuratePass_p90', 'TotalShots_p90', 'ShotOnTarget_p90',
       'penaltyWon', 'punches',
       'totalKeeperSweeper', 'accurateKeeperSweeper', 'penaltyMiss',
       'penaltySave', 'clearanceOffLine']:
    
    
    stat_val = home_df[home_df['player'] == player][stat].iloc[0]

    if stat_val > 0.0:

        home_df.loc[:, 'rank'] = home_df[stat].rank(ascending=False, method='min')

        #rank df
        rdf = home_df[['player', stat, 'rank']].sort_values(by='rank', ascending=True)

        pdf = rdf[rdf['player'] == player]

        rank = pdf['rank'].iloc[0]

        ranks.append({"stat": stat, "value": stat_val, "rank": rank})

        # print(rank)

ranks = pd.DataFrame(ranks).sort_values(by="rank")
ranks

# df[['player', 'possessionLostCtrl']].sort_values(by='possessionLostCtrl')
    
        

Luis Castro ranks


,stat,value,rank
16,TotalShots,2.0,1.0
15,ShotOnTarget,1.0,1.0
14,bigChanceCreated,1.0,1.0
5,totalCross,5.0,1.0
6,accurateCross,1.0,1.0
8,duelWon,7.0,1.0
11,keyPass,2.0,1.0
9,totalTackle,2.0,2.0
13,wonContest,1.0,2.0
12,totalContest,1.0,2.0


Let's create some subjective stats that are more impressive to prioritise those

In [116]:
level1 = ['goals', 'goalAssist', 'ShotOnTarget', 'keyPass', 'TotalShots', 'bigChanceCreated']
level2 = ['accurateCross', 'totalCross', 'duelWon', 'touches', 'aerialWon' ]

top5 = []

for i, row in ranks.iterrows():
    if len(top5) == 5:
        break
    stat = row['stat']
    if stat in level1:
        print(stat, row['value'])
        top5.append(row)
    elif stat in level2:
        print(stat, row['value'])
        top5.append(row)
        
pd.DataFrame(top5)

TotalShots 2.0
ShotOnTarget 1.0
bigChanceCreated 1.0
totalCross 5.0
accurateCross 1.0


,stat,value,rank
16,TotalShots,2.0,1.0
15,ShotOnTarget,1.0,1.0
14,bigChanceCreated,1.0,1.0
5,totalCross,5.0,1.0
6,accurateCross,1.0,1.0
